# Phase 2: Named Entity Recognition (NER)
## BBC News Business Category

---

### What is this notebook doing?

In Phase 1, we used BGE-M3 embeddings → UMAP → HDBSCAN → BERTopic to discover
**what** each article is about,assigning every article a sub-category label
like "Corporate Earnings" or "Fiscal Policy".

Phase 2 asks a different question entirely:

> **Who and what are actually mentioned inside these articles?**

That is the job of Named Entity Recognition (NER). Instead of looking at the
shape of a whole document, NER reads every sentence and tags individual words
and phrases by type:

| Entity Type | Meaning             | Example               |
|-------------|---------------------|-----------------------|
| `PERSON`    | People's names      | Alan Greenspan        |
| `ORG`       | Organisations       | Federal Reserve       |
| `GPE`       | Countries / Cities  | United States         |
| `MONEY`     | Monetary values     | $1.13bn               |
| `DATE`      | Dates / time refs   | December 2004         |
| `NORP`      | Nationalities       | European              |

---

### How does Phase 2 fit into the full architecture?
```
Phase 1 Output (final_df)
  filename | sub_category | confidence_score | preview
       ↓
Phase 2 — NER runs on the same article text
       ↓
  Extract → PERSON, ORG, GPE, MONEY, DATE entities
  Classify → assign a role to each PERSON (CEO, Politician, Economist...)
       ↓
Phase 2 Output — enriched_df
  filename | sub_category | people | people_roles | organisations | locations
```

BERTopic and NER are **complementary**, not competing.
BERTopic tells you the topic. NER tells you the cast of characters inside it.

---

### Tool: spaCy
We use `spaCy` with the `en_core_web_sm` English pipeline — a model trained on
web text and news data, making it well suited for BBC articles.

## Imports & Environment

We import four things here:

- **spacy** — the NER engine that will read and tag our articles
- **pandas** — to build and manipulate DataFrames
- **Counter** — from Python's built-in `collections` module; lets us count
  how often each entity (e.g. a person's name) appears across all articles
- **display** — from IPython; renders DataFrames as formatted tables in Colab

In [ ]:
 # spacy is our NER engine
import spacy

# pandas for DataFrame manipulation
import pandas as pd

# collections.Counter lets us count how often each entity appears
from collections import Counter

# We'll display rich tables in Colab
from IPython.display import display


## Load the spaCy Model

`spacy.load("en_core_web_sm")` reads the trained English pipeline from disk
and returns an `nlp` object.

Calling `nlp(text)` on any string runs it through a sequence of components:
```
text → tok2vec → tagger → parser → ner → attribute_ruler → lemmatizer → Doc
```



In [ ]:

#  Load the spaCy English pipeline

# spacy.load() reads the trained en_core_web_sm model from disk.
# This model was trained on web text and news perfect for BBC articles.
# The `nlp` object is a callable pipeline: nlp(text) → processed Doc.

nlp = spacy.load("en_core_web_sm")

print("spaCy model loaded:", nlp.meta["name"])
print("Pipeline components:", nlp.pipe_names)
# You'll see: ['tok2vec', 'tagger', 'parser', 'ner', 'attribute_ruler', 'lemmatizer']
# The 'ner' component is what we care about here.

## Load Phase 1 Output

We load the `final_Business_Subcategories.csv` we produced at the end of
Phase 1. This gives us the `preview` column the full article text which
is what NER will run on.

We also get `sub_category` for free, so our enriched output will carry both
the topic label (from BERTopic now human labelled) and the entity data (from NER) in one place.


In [ ]:
# Load the CSV we saved at the end of Phase 1.
# Adjust this path to wherever you saved your final_df.
final_df = pd.read_csv('Dataframe/bbc-nlp-classifier/New_final_Business_Subcategories.csv')

print(f"Loaded {len(final_df)} articles")
print(final_df[['filename', 'sub_category', 'confidence_score']].head())

## NER Extraction Function

This is the core of Phase 2. The function `extract_entities()` takes a single
article's text, passes it through the spaCy pipeline, and returns a dictionary
of entity lists grouped by type.

**How it works line by line:**

1. `doc = nlp(text)`  runs the full pipeline and returns a `Doc` object
2. `doc.ents` a tuple of every entity spaCy detected in the text
3. We loop over each entity (`ent`) and check its `label_`
4. If the label is one we care about, we append `ent.text` to the right list
5. `.strip()` cleans any accidental whitespace from the entity string

**Why only 6 entity types?**
spaCy's `en_core_web_sm` recognises 18 entity types in total (including things
like `WORK_OF_ART`, `LAW`, `LANGUAGE`). We filter to the 6 most meaningful
for news articles people, organisations, places, money, dates, and groups.

In [ ]:
# NER extraction function

def extract_entities(text):
    """
    Run spaCy NER on a single article and return a dict of entity lists.

    nlp(text) runs the full pipeline and returns a Doc object.
    doc.ents is a tuple of Span objects each Span is one detected entity.
    span.text   → the actual string  e.g. "Alan Greenspan"
    span.label_ → the entity type   e.g. "PERSON"
    """

    # Pass the text through the spaCy pipeline
    doc = nlp(text)

    # Initialise empty lists for each entity type we care about
    entities = {
        "PERSON": [],   # people's names
        "ORG":    [],   # organisations, companies
        "GPE":    [],   # geopolitical entities: countries, cities
        "MONEY":  [],   # monetary values
        "DATE":   [],   # dates and time references
        "NORP":   [],   # nationalities and political/religious groups
    }

    # Loop over every entity spaCy found
    for ent in doc.ents:
        # Only keep entity types we defined above
        if ent.label_ in entities:
            # .strip() removes any accidental leading/trailing whitespace
            entities[ent.label_].append(ent.text.strip())

    return entities

## Role Classifier

spaCy tells us **who** is mentioned but not **what they do**.
The assignment specifically asks us to identify the roles of named people
Politicians, CEOs, Economists, etc.

We solve this with a **context window approach**:

1. Find the position of the person's name in the article text
2. Extract a 300 character window of text surrounding that name
   (150 characters before, 150 after) this is the local context
3. Search that window for job-title keywords from our `ROLE_KEYWORDS` dict
4. Return the first matching role, or `"Unknown"` if none match

**Why a context window instead of a trained model?**
A trained role-classification model would need labelled data we don't have.
The context window approach is lightweight, fully interpretable, and works
well for structured news writing journalists almost always introduce a
person with their title nearby e.g. *"Federal Reserve chairman Alan Greenspan"*.

In [ ]:
# NER extraction function

def extract_entities(text):
    """
    Run spaCy NER on a single article and return a dict of entity lists.

    nlp(text) runs the full pipeline and returns a Doc object.
    doc.ents is a tuple of Span objects each Span is one detected entity.
    span.text   → the actual string  e.g. "Alan Greenspan"
    span.label_ → the entity type   e.g. "PERSON"
    """

    # Pass the text through the spaCy pipeline
    doc = nlp(text)

    # Initialise empty lists for each entity type we care about
    entities = {
        "PERSON": [],   # people's names
        "ORG":    [],   # organisations, companies
        "GPE":    [],   # geopolitical entities: countries, cities
        "MONEY":  [],   # monetary values
        "DATE":   [],   # dates and time references
        "NORP":   [],   # nationalities and political/religious groups
    }

    # Loop over every entity spaCy found
    for ent in doc.ents:
        # Only keep entity types we defined above
        if ent.label_ in entities:
            # .strip() removes any accidental leading/trailing whitespace
            entities[ent.label_].append(ent.text.strip())

    return entities

## Role Classifier

spaCy tells us **who** is mentioned but not **what they do**.
The assignment specifically asks us to identify the roles of named people
Politicians, CEOs, Economists, etc.

We solve this with a **context window approach**:

1. Find the position of the person's name in the article text
2. Extract a 300 character window of text surrounding that name
   (150 characters before, 150 after) this is the local context
3. Search that window for job-title keywords from our `ROLE_KEYWORDS` dict
4. Return the first matching role, or `"Unknown"` if none match

**Why a context window instead of a trained model?**
A trained role-classification model would need labelled data we don't have.
The context window approach is lightweight, fully interpretable, and works
well for structured news writing journalists almost always introduce a
person with their title nearby e.g. *"Federal Reserve chairman Alan Greenspan"*.

In [ ]:
#Role classifier
# BUSINESS FALSE_PERSON_FILTER

BUSINESS_FALSE_PERSONS = {
    # Brand/company names spaCy tags exclusively as PERSON
    "pernod ricard", "diageo", "glenmorangie", "happoshu", "aston martin",
    "bertelsmann", "glaxosmithkline", "man utd", "yuganskneftegas",
    "doubleclick", "altria", "ladybird", "biogen idec",
    # Single-word tokens passing through
    "beer", "euro", "daily", "bush", "may", "castro", "rotterdam",
    "buenos aires", "sunovate", "vivaldi", "tamil nadu",
    # Number strings spaCy misreads as names
    "6.2bn", "200bn", "280bn",
    # Musicians/entertainers appearing in business articles
    "joss stone", "franz ferdinand", "justin timberlake", "ray charles",
    "destiny",
}

# Add to NAME_OVERRIDES — business figures with no keyword context
NAME_OVERRIDES = {
    "gianni agnelli":  "CEO/Executive",   # Fiat chairman
    "shigeo morioka":  "CEO/Executive",
}
# Role keyword dictionary: role label → list of trigger words
ROLE_KEYWORDS = {
    "Politician":     ["minister", "prime minister", "president", "senator",
                       "chancellor", "secretary", "governor", "congressman",
                       "mp", "lawmaker", "official", "treasury", "finance minister",
                       "trade minister", "central bank", "regulator"],

    "CEO/Executive":  ["chief executive", "ceo", "chairman", "president",
                       "director", "managing director", "chief financial",
                       "cfo", "vice president", "head of", "founder",
                       "co-founder", "chief operating", "coo", "partner",
                       "executive", "officer", "boss", "leader", "chief"],

    "Economist":      ["economist", "analyst", "strategist", "researcher",
                       "professor", "academic", "forecaster", "advisor",
                       "adviser", "commentator", "expert", "consultant",
                       "think tank", "institute", "fund manager", "investor"],

    "Lawyer":         ["lawyer", "attorney", "prosecutor", "counsel",
                       "solicitor", "barrister", "judge", "legal",
                       "court", "trial", "charged", "convicted"],

    "Spokesperson":   ["spokesman", "spokeswoman", "spokesperson",
                       "said", "told", "added", "warned", "stated",
                       "confirmed", "announced", "claimed", "argued"],
}

def safe_join(items, placeholder="None"):
    if isinstance(items, list):
        cleaned = [str(i).strip() for i in items if str(i).strip()]
        return ', '.join(cleaned) if cleaned else placeholder
    val = str(items).strip()
    val = val.strip("[]'\"")
    if not val or ('No ' in val and 'found' in val):
        return placeholder
    return val

def safe_roles_str(roles_dict, placeholder="None"):
    if not roles_dict:
        return placeholder
    return ', '.join(f"{k}: {v}" for k, v in roles_dict.items())

def classify_role(person_name, full_text):
    """
    Given a person's name and the full article text, find the sentence(s)
    that mention them and look for role keywords nearby.

    Returns the matched role label, or 'Unknown' if none found.
    """
    if person_name.lower() in NAME_OVERRIDES:
        return NAME_OVERRIDES[person_name.lower()]

    # Lowercase everything so matching is case-insensitive
    text_lower = full_text.lower()
    name_lower = person_name.lower()

    # Find the position of this person's name in the text
    name_pos = text_lower.find(name_lower)

    # If the name isn't found (edge case), return Unknown
    if name_pos == -1:
        return "Unknown"

    # Extract a 300-character window around the name.
    # max(0, ...) prevents a negative start index.
    # min(len(...), ...) prevents going past the end of the text.
    start = max(0, name_pos - 250)
    end   = min(len(text_lower), name_pos + 250)
    context = text_lower[start:end]

    # Check each role's keywords against the context window
    for role, keywords in ROLE_KEYWORDS.items():
        for kw in keywords:
            if kw in context:
                return role  # Return the first matching role

    return "Unknown"

## Run NER Across All Articles

Now we apply both functions to the full DataFrame.

For each article we:
1. Call `extract_entities()` to get all entity lists
2. Loop over each `PERSON` entity and call `classify_role()` on it
3. Deduplicate people using a dict — same name can appear multiple times
   in one article; we keep the last-seen role assignment
4. Package everything into a result dict and append to a list

**Why build a list of dicts and convert at the end?**
Appending rows to a DataFrame inside a loop with `.loc` or `.append()` is
very slow in pandas — it copies the entire DataFrame on every iteration.
Building a plain Python list of dicts first and calling `pd.DataFrame()`
once at the end is the correct, efficient pattern.

In [ ]:
#Role classifier
# BUSINESS FALSE_PERSON_FILTER

BUSINESS_FALSE_PERSONS = {
    # Brand/company names spaCy tags exclusively as PERSON
    "pernod ricard", "diageo", "glenmorangie", "happoshu", "aston martin",
    "bertelsmann", "glaxosmithkline", "man utd", "yuganskneftegas",
    "doubleclick", "altria", "ladybird", "biogen idec",
    # Single-word tokens passing through
    "beer", "euro", "daily", "bush", "may", "castro", "rotterdam",
    "buenos aires", "sunovate", "vivaldi", "tamil nadu",
    # Number strings spaCy misreads as names
    "6.2bn", "200bn", "280bn",
    # Musicians/entertainers appearing in business articles
    "joss stone", "franz ferdinand", "justin timberlake", "ray charles",
    "destiny",
}

# Add to NAME_OVERRIDES — business figures with no keyword context
NAME_OVERRIDES = {
    "gianni agnelli":  "CEO/Executive",   # Fiat chairman
    "shigeo morioka":  "CEO/Executive",
}
# Role keyword dictionary: role label → list of trigger words
ROLE_KEYWORDS = {
    "Politician":     ["minister", "prime minister", "president", "senator",
                       "chancellor", "secretary", "governor", "congressman",
                       "mp", "lawmaker", "official", "treasury", "finance minister",
                       "trade minister", "central bank", "regulator"],

    "CEO/Executive":  ["chief executive", "ceo", "chairman", "president",
                       "director", "managing director", "chief financial",
                       "cfo", "vice president", "head of", "founder",
                       "co-founder", "chief operating", "coo", "partner",
                       "executive", "officer", "boss", "leader", "chief"],

    "Economist":      ["economist", "analyst", "strategist", "researcher",
                       "professor", "academic", "forecaster", "advisor",
                       "adviser", "commentator", "expert", "consultant",
                       "think tank", "institute", "fund manager", "investor"],

    "Lawyer":         ["lawyer", "attorney", "prosecutor", "counsel",
                       "solicitor", "barrister", "judge", "legal",
                       "court", "trial", "charged", "convicted"],

    "Spokesperson":   ["spokesman", "spokeswoman", "spokesperson",
                       "said", "told", "added", "warned", "stated",
                       "confirmed", "announced", "claimed", "argued"],
}

def safe_join(items, placeholder="None"):
    if isinstance(items, list):
        cleaned = [str(i).strip() for i in items if str(i).strip()]
        return ', '.join(cleaned) if cleaned else placeholder
    val = str(items).strip()
    val = val.strip("[]'\"")
    if not val or ('No ' in val and 'found' in val):
        return placeholder
    return val

def safe_roles_str(roles_dict, placeholder="None"):
    if not roles_dict:
        return placeholder
    return ', '.join(f"{k}: {v}" for k, v in roles_dict.items())

def classify_role(person_name, full_text):
    """
    Given a person's name and the full article text, find the sentence(s)
    that mention them and look for role keywords nearby.

    Returns the matched role label, or 'Unknown' if none found.
    """
    if person_name.lower() in NAME_OVERRIDES:
        return NAME_OVERRIDES[person_name.lower()]

    # Lowercase everything so matching is case-insensitive
    text_lower = full_text.lower()
    name_lower = person_name.lower()

    # Find the position of this person's name in the text
    name_pos = text_lower.find(name_lower)

    # If the name isn't found (edge case), return Unknown
    if name_pos == -1:
        return "Unknown"

    # Extract a 300-character window around the name.
    # max(0, ...) prevents a negative start index.
    # min(len(...), ...) prevents going past the end of the text.
    start = max(0, name_pos - 250)
    end   = min(len(text_lower), name_pos + 250)
    context = text_lower[start:end]

    # Check each role's keywords against the context window
    for role, keywords in ROLE_KEYWORDS.items():
        for kw in keywords:
            if kw in context:
                return role  # Return the first matching role

    return "Unknown"

## Run NER Across All Articles

Now we apply both functions to the full DataFrame.

For each article we:
1. Call `extract_entities()` to get all entity lists
2. Loop over each `PERSON` entity and call `classify_role()` on it
3. Deduplicate people using a dict — same name can appear multiple times
   in one article; we keep the last-seen role assignment
4. Package everything into a result dict and append to a list

**Why build a list of dicts and convert at the end?**
Appending rows to a DataFrame inside a loop with `.loc` or `.append()` is
very slow in pandas — it copies the entire DataFrame on every iteration.
Building a plain Python list of dicts first and calling `pd.DataFrame()`
once at the end is the correct, efficient pattern.

## Merge NER Results into the Main DataFrame

We now join `ner_df` back onto `final_df` using `filename` as the shared key.



In [ ]:
# Run NER across the full DataFrame with org-name filtering

# clean_entity_name(), safe_join(), safe_roles_str()
# and the category-specific FALSE_PERSON_FILTER patch above

import re

def clean_entity_name(name):
    """Strip possessives, trailing dashes, trailing punctuation."""
    name = name.strip()
    name = re.sub(r"'s$", "", name)       # "Bekele's" → "Bekele"
    name = re.sub(r"'\s*$", "", name)     # "Kenteris'" → "Kenteris"
    name = re.sub(r"\s*-\s*$", "", name)  # "Gardener -" → "Gardener"
    name = name.rstrip(",'.")
    return name.strip()

results = []

for _, row in final_df.iterrows():

    text     = row['preview']
    entities = extract_entities(text)

    org_names_lower = {org.lower() for org in entities["ORG"]}

    # Clean and filter PERSON entities
    seen = set()
    filtered_people = []
    for raw_name in entities["PERSON"]:
        name = clean_entity_name(raw_name)
        if (name.lower() not in org_names_lower
                and name.lower() not in BUSINESS_FALSE_PERSONS
                and len(name) > 3
                and ' ' in name
                and name not in seen):
            filtered_people.append(name)
            seen.add(name)

    #  Classify roles
    people_dict = {}
    for name in filtered_people:
        people_dict[name] = classify_role(name, text)

    #  Filter money_refs to genuine monetary values only
    clean_money = [
        m for m in entities.get("MONEY", [])
        if any(c in m for c in [
            '£', '$', '€', 'bn', 'mn', 'million', 'billion',
            'trillion', '%', 'per cent', 'm (', 'n ('
        ])
    ]

    results.append({
        "filename":      row["filename"],
        "sub_category":  row["sub_category"],
        "people":        safe_join(list(people_dict.keys())),
        "people_roles":  safe_roles_str(people_dict),
        "organisations": safe_join(list(set(entities["ORG"]))),
        "locations":     safe_join(list(set(entities["GPE"]))),
        "money_refs":    safe_join(clean_money),
        "dates":         safe_join(list(set(entities.get("DATE", [])))),
    })

ner_df = pd.DataFrame(results)
print(f"NER complete. Shape: {ner_df.shape}")
display(ner_df.head(5))

## Merge NER Results into the Main DataFrame

We now join `ner_df` back onto `final_df` using `filename` as the shared key.



In [ ]:
# Merge NER results back into the main DataFrame

# Join on 'filename' the shared key between both DataFrames.
# how='left' keeps every row from final_df and attaches NER columns
# where a filename match exists.

enriched_df = final_df.merge(
    ner_df[['filename', 'people', 'people_roles', 'organisations', 'locations', 'money_refs']],
    on='filename',
    how='left'
)

print(f"Enriched DataFrame shape: {enriched_df.shape}")
print("\nColumns:", enriched_df.columns.tolist())
display(enriched_df[['filename', 'sub_category', 'people', 'people_roles', 'organisations', 'locations']].head(5))

## Summary: Most Mentioned People and Their Roles

This cell answers a key question:

> *Who are the most prominent people across BBC Business articles,
> and what roles do they hold?*

**How it works:**
1. We flatten all `people_roles` dicts from `ner_df` into one master list
   of `(name, role)` tuples one entry per person per article
2. `Counter` counts how many articles each name appears in
3. `.most_common(20)` returns the top 20 names by appearance count
4. We build a clean summary DataFrame with name, role, and appearance count

In [ ]:
# Flatten all people_roles dicts into one master list of (name, role) tuples
all_people = []
for roles_str in ner_df['people_roles']:
    # Parse the string back into a dictionary-like structure
    current_roles_dict = {}
    if roles_str and roles_str != "None": # Check for non-empty and not "None" placeholder
        pairs = roles_str.split(', ')
        for pair in pairs:
            if ': ' in pair: # Ensure it's a valid 'name: role' pair
                name, role = pair.split(': ', 1) # Split only on the first colon to handle roles with colons
                current_roles_dict[name.strip()] = role.strip()

    for name, role in current_roles_dict.items():
        all_people.append((name, role))

# Count by name how many articles does each person appear in?
name_counts = Counter([name for name, role in all_people])

# Build a clean lookup dict: name → role (last seen wins on duplicates)
name_to_role = {name: role for name, role in all_people}

# Build the summary DataFrame from the top 20 names
people_summary = pd.DataFrame([
    {
        "name":        name,
        "role":        name_to_role.get(name, "Unknown"),
        "appearances": count
    }
    for name, count in name_counts.most_common(20)
])

print("\nTOP 20 MOST MENTIONED PEOPLE FOR BBC BUSINESS")
print("=" * 55)
display(people_summary)